# M8 · v1.0 · ¿Lo liberarías?

Evidencia para decidir si gente real podría usarlo: **traza**, **costo**, **protección con test** y una práctica de producción. Es tu **v1.0**.

**Para qué te sirve:** saber si lo podés dejar en manos de otros: cuánto cuesta cada respuesta, qué pasó en cada paso cuando falla (como el historial de un refresh de Power BI) y que tu peor error no se repita.

**La tarea:** 1. **Escribí** `v10_produccion.py` · 2. **El guardrail, con test** · 3. **Operá y costeá** · 4. **Go / no-go**.

- Corré las celdas en orden con **Shift + Enter**. La clave: **Secretos** (la llave de la izquierda) → `GOOGLE_API_KEY`.
- Como en tu repo, cada versión es un **archivo**. Cada paso tiene el fragmento en una celda de texto: **escribilo** en la celda
  de abajo, debajo de su primera línea (`%%writefile …` guarda la celda como archivo; con `-a`, lo suma al final). Después, un **chequeo**.
- **Con tu asistente de IA** (el que usás, o Gemini dentro de Colab): pegale el fragmento y pedile «explicame cada renglón». Si sale un error, pegáselo.
- ¿Tenés tus versiones anteriores? Subilas con la carpeta de la izquierda **después** de la preparación: reemplazan a las de ejemplo.

v1.0 es tu versión de siempre con lo que pide un tablero que usa gente real: un registro de cada consulta (la traza, como el log de actualizaciones de Power BI), su costo, y que un dato personal no se escape.

<details><summary><b>¿Sale un error?</b></summary>

- **«No pude leer el secreto GOOGLE_API_KEY»** o **`API key not valid`:** revisá el nombre del secreto (exacto, en mayúsculas),
  que «Acceso del notebook» esté activado y que la clave esté bien copiada.
- **`503 UNAVAILABLE`** («high demand»): el modelo está saturado, no es tu código. Esperá un minuto y volvé a correr.
- **`429`** (*quota*, *RESOURCE_EXHAUSTED*): se terminó la cuota gratis del día o del minuto. Esperá o seguí mañana.
- **`NameError`:** salteaste una celda o Colab se reinició. Menú *Entorno de ejecución* → *Run before*.
- **`buscar_web` da ERROR** (`RatelimitException`, `TimeoutException`): DuckDuckGo cortó. Para medir, `WEB = "grabada"`.
</details>

## 1 · Preparar (una vez, un par de minutos)

Instala, lee tu clave, baja lo de la plantilla. Después pegás tus archivos de antes.

In [ ]:
%pip install -q langchain==1.4.6 langchain-google-genai==4.4.2 chromadb==1.5.9 langchain-text-splitters==1.1.3 ddgs==9.16.0 pytest
import os, subprocess, sys, urllib.request
from pathlib import Path

try:
    from google.colab import userdata                      # en Colab: la clave sale de los Secretos
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass                                                   # fuera de Colab: GOOGLE_API_KEY ya tiene que estar
except Exception as error:                                 # no hay secreto con ese nombre o le falta el acceso
    print("No pude leer el secreto GOOGLE_API_KEY:", type(error).__name__)
os.environ["WEB"] = "grabada"  # la web de herramientas.py: "real", "grabada" o "caida" (en tu repo, en config.toml)

# Lo que en tu repo hace llm.py (acá no hay config.toml)
Path("llm.py").write_text("""from pathlib import Path
from langchain.chat_models import init_chat_model
RAIZ = Path(__file__).resolve().parent
def cargar_modelo():
    return init_chat_model("google_genai:gemini-3.5-flash-lite", max_output_tokens=300)
def config():
    return {}
""", encoding="utf-8")

# De la plantilla: los documentos de ejemplo, las preguntas de prueba, herramientas.py y la web grabada
PLANTILLA = "https://raw.githubusercontent.com/JuanFKurucz/ucu-la-project/main/"
for archivo in ["corpus/langchain__short-term-memory.md", "corpus/langgraph__interrupts.md", "corpus/langgraph__persistence.md",
                "corpus/langgraph__thinking-in-langgraph.md", "corpus/langgraph__use-subgraphs.md", "datos/m4_suite.jsonl",
                "datos/m6_seleccion.jsonl", "fixtures/web.json", "herramientas.py"]:
    Path(archivo).parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(PLANTILLA + archivo, archivo)


def correr(archivo):
    """Como «uv run python archivo» en tu repo: lo corre y muestra lo que imprime."""
    r = subprocess.run([sys.executable, archivo], capture_output=True, text=True, encoding="utf-8",
                       env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    print(r.stdout[-4000:], r.stderr[-2000:] if r.returncode else "")
    return r.stdout


print("Listo." if os.environ.get("GOOGLE_API_KEY") else "Falta la clave: Secretos → GOOGLE_API_KEY.")

### Tus archivos de antes

Pegá en cada celda, **debajo de la primera línea**, tu archivo tal como lo tenés en tu repo (o copiá la celda de tu cuaderno anterior) y corré la celda:

- `v02_rag.py`, de la práctica M4
- `v04_agente.py`, de la práctica M6
- `v05_equipo.py`, de la práctica M7
- `medir.py`, de la práctica M6

¿No los tenés? Están completos en esas prácticas.

In [ ]:
%%writefile v02_rag.py
# Pegá acá tu v02_rag.py de M4

In [ ]:
%%writefile v04_agente.py
# Pegá acá tu v04_agente.py de M6

In [ ]:
%%writefile v05_equipo.py
# Pegá acá tu v05_equipo.py de M7

In [ ]:
%%writefile medir.py
# Pegá acá tu medir.py de M6

In [ ]:
for archivo, debe in [("v02_rag.py", "def buscar"), ("v04_agente.py", "def responder"), ("v05_equipo.py", "def responder"), ("medir.py", "responder(")]:
    assert debe in Path(archivo).read_text(encoding="utf-8"), f"Falta tu {archivo}: pegalo en su celda y corré."
print("Listo: tus archivos de antes están.")

## 2 · Tu v1.0

Envuelve tu última versión; tapa los emails, calcula el costo, deja la traza y, si algo falla, avisa en vez de inventar. Debajo de `%%writefile v10_produccion.py`:

```python
import datetime, json, os, re
import v05_equipo as base  # o v04_agente, si en M7 ganó el agente solo
PRECIO_ENTRADA, PRECIO_SALIDA, PRECIO_FECHA = 0.30, 2.50, "19/10"  # US$ por millón de tokens, de la página del proveedor
def tapar(texto: str) -> str:  # el guardrail: un email no llega al modelo ni a la traza
    return re.sub(r"[\w.+-]+@[\w-]+(\.[\w-]+)+", "[email]", texto)
def responder(pregunta: str) -> dict:
    try:
        r = base.responder(tapar(pregunta))
    except Exception as error:  # fallo seguro: un aviso claro, nunca una respuesta inventada
        r = {"respuesta": "No pude responder ahora. Probá en unos minutos.", "error": type(error).__name__,
             "tokens_entrada": 0, "tokens_salida": 0}
    r["costo_usd"] = (r["tokens_entrada"] * PRECIO_ENTRADA + r["tokens_salida"] * PRECIO_SALIDA) / 1_000_000
    os.makedirs("logs", exist_ok=True)
    with open("logs/traza.jsonl", "a", encoding="utf-8") as traza:  # una línea por pregunta
        fila = {"cuando": datetime.datetime.now().isoformat(timespec="seconds"), "pregunta": tapar(pregunta), "precio_del": PRECIO_FECHA, **r}
        traza.write(json.dumps(fila, ensure_ascii=False) + "\n")
    return r
if __name__ == "__main__": print(responder("¿Qué hace interrupt()? Respondeme a ana.perez@ucu.edu.uy"))
```

`re.sub(patrón, "[email]", texto)` es un `REGEXP_REPLACE` de SQL: el patrón describe «letras, @, dominio con puntos».

In [ ]:
%%writefile v10_produccion.py
# v1.0: tapar, fallo seguro, costo y traza

In [ ]:
import json
correr("v10_produccion.py")
fila = json.loads(Path("logs/traza.jsonl").read_text(encoding="utf-8").splitlines()[-1])
print(fila)
assert "@" not in fila["pregunta"] and "costo_usd" in fila, "En la traza, la pregunta va con [email] y la fila lleva costo_usd"

**Qué deberías ver:** la respuesta con `costo_usd`; en la traza, la pregunta con `[email]`, las herramientas, los tokens y el costo.

**Primer control · 10–15 min:** en la traza, ningún email y el costo de la consulta.

## 3 · El guardrail, con test

Debajo de `%%writefile tests/test_produccion.py` (no gasta: el segundo simula la caída):

```python
import v10_produccion
def test_un_email_no_pasa():
    assert v10_produccion.tapar("soy ana.perez@ucu.edu.uy, ¿me ayudás?") == "soy [email], ¿me ayudás?"
def test_si_falla_avisa(monkeypatch):
    monkeypatch.setattr(v10_produccion.base, "responder", lambda pregunta: 1 / 0)  # solo en este test: tu versión «se cae»
    assert "No pude responder" in v10_produccion.responder("hola")["respuesta"]
```

In [ ]:
%%writefile tests/test_produccion.py
# tus tests: el email y la caída

In [ ]:
!python -m pytest -q -p no:cacheprovider tests

**Qué deberías ver:** todos `passed`, ningún `failed`.

**Tu peor fallo:** sumá un test con un caso tuyo (otro dato personal, una pregunta que lo hizo inventar) a la celda de arriba y corré de nuevo. ¿`failed`? Anotá qué no atrapa el guardrail.

## 4 · Operá y costeá

Con la web grabada: `medir.py` con tu versión cambiada por `v10_produccion` (en el `import` y en la llamada) y, al final, el costo (es un mes a 1000 por día):

```python
import json, pandas as pd, v10_produccion
suite = [json.loads(linea) for linea in open("datos/m6_seleccion.jsonl", encoding="utf-8") if linea.strip()]
tabla = pd.DataFrame([{**caso, "esperadas": caso["herramientas"], **v10_produccion.responder(caso["pregunta"])} for caso in suite])
tabla["eligio_bien"] = [set(e) == set(h) for e, h in zip(tabla["esperadas"], tabla["herramientas"])]
print(tabla[["id", "tipo", "eligio_bien", "herramientas", "llamadas"]].to_string())
print("US$ por consulta:", tabla["costo_usd"].mean(), "· por mes:", tabla["costo_usd"].mean() * 1000 * 30)
tabla.to_csv("logs/medir.csv", index=False)  # para la tabla dinámica de tus fallas, en Excel o Power BI
```

In [ ]:
%%writefile medir_v10.py
# medir.py con v10_produccion y el costo

In [ ]:
correr("medir_v10.py")

¿Cuánto sería con tu uso real?

1. **Tu práctica de producción** (una): trazas, costos, guardrail, caché o evaluación continua.
   Caché en tres líneas, arriba de `v10_produccion.py`: `from langchain_core.caches import InMemoryCache`,
   `from langchain_core.globals import set_llm_cache` y `set_llm_cache(InMemoryCache())`. Vive mientras corre el programa:
   repetí una pregunta y medí `latencia_s`.
2. **Go / no-go** en media página: «**No-go por ahora.** Suite: x/n. Costo: US$ … por consulta y por mes (precio del dd/mm). Test: pasa. Sigue fallando: … Probaría: …».

## Entregar (entrega final, hasta el dom 22/11, 23:59)

- **Tus archivos a tu repo:** en la carpeta de la izquierda, clic derecho → *Descargar* en `v10_produccion.py` y `tests/test_produccion.py`; copialos a la carpeta del proyecto, *commit* y *push*.
- **Evidencia** (una celda de texto al final, acá mismo, o en `evidencia/v1.0.md`): un extracto de la traza (sin claves ni datos personales), el costo con fecha, el test y la decisión.

**Microdefensa:** ¿Qué te impide liberarlo hoy? ¿Qué prueba lo resolvería? Confianza del 1 («no lo sabría explicar») al 4 («lo explico solo»).